# Animal Shelter: adoption drop-offs (conversion problem)

***
**The problem**
People show interest...but don't complete the adoption.  Why are people dropping off before adoption?

**Why it matters**

 - Lost opportunities for animals to be rehomed
 - Wasted staff time

**Analytics angle**

 - Funnel analyis:
    - Website visit -> enquiry -> visit -> adoption
 - Where are people dropping off?
 - Does response time affect success?
 - Are certain animals abandoned more in the process?

**Data**

 - animals.csv - basic information about each animal
 - enquiries.csv - every time someone expresses interest
 - visits.csv - did they actually come to meet the animal?
 - adoptions.csv - final outcome

***
### Notebook 1: Data Preparation

1. Reading CSV files
2. Adoptions DataFrame
3. Animals DataFrame
4. Enquiries DataFrame
5. Visits DataFrame
6. Create SQLite database
7. Load tables into SQL
8. Cleaning log
9. Cleaning data using SWL
10. Re-load DataFrames into SWL
11. Create an adoption analysis dataset
12. Write adoption analysis dataset to CSV
13. Initial queries to sense-check data

***
## Reading csv files

In [783]:
import pandas as pd

adoptions = pd.read_csv("/Users/lisa/Documents - Mac/Coding Projects/animal-shelter-project/data/adoptions_v2.csv")

animals = pd.read_csv("/Users/lisa/Documents - Mac/Coding Projects/animal-shelter-project/data/animals_v2.csv")

enquiries = pd.read_csv("/Users/lisa/Documents - Mac/Coding Projects/animal-shelter-project/data/enquiries_v2.csv")

visits = pd.read_csv("/Users/lisa/Documents - Mac/Coding Projects/animal-shelter-project/data/visits_v2.csv")


***
## adoptions

In [784]:
adoptions.head()

,adoption_id,enquiry_id,animal_id,customer_id,adoption_date,fee_paid,returned_within_30_days,return_date
0,AD0001,E0175,A136,C0161,2025/04/07,120.0,No,NaN
1,AD0002,E0274,A142,C0342,2025/01/24,75.0,No,NaN
2,AD0003,E0062,A022,C0263,2025/02/06,150.0,no,NaN
3,AD0004,E0097,A086,C0118,23/04/2025,100.0,Yes,30 April 2025
4,AD0005,E0283,A117,C0058,10/03/2025,200.0,No,NaN


In [785]:
adoptions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113 entries, 0 to 112
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   adoption_id              113 non-null    object 
 1   enquiry_id               113 non-null    object 
 2   animal_id                113 non-null    object 
 3   customer_id              113 non-null    object 
 4   adoption_date            113 non-null    object 
 5   fee_paid                 107 non-null    float64
 6   returned_within_30_days  106 non-null    object 
 7   return_date              16 non-null     object 
dtypes: float64(1), object(7)
memory usage: 7.2+ KB


In [786]:
adoptions.describe()

,fee_paid
count,107.000000
mean,146.588785
std,57.015382
min,75.000000
25%,95.000000
50%,120.000000
75%,200.000000
max,250.000000


***
**adoption_id**

We begin by checking to see if adoption_ids are individual, i.e., that we have no duplicated entries:

In [787]:
adoptions['adoption_id'].duplicated().sum()

8

We see that there are 8 duplicated entries, so we will undergo our process to determine if these are full duplicated records or if there are any anomalies worthy of further investigation:

In [788]:
duplicate_adoptions = (
    adoptions[
    adoptions['adoption_id'].duplicated(keep=False)
    ]
    .sort_values('adoption_id')
)

In [789]:
exact_duplicated_adoptions = adoptions.duplicated().sum()

duplicate_adoption_rows = (
    adoptions['adoption_id'].duplicated().sum()
)
conflicting_duplicated_adoptions = (
    duplicate_adoption_rows - exact_duplicated_adoptions
)
print(f"Exact duplicates: {exact_duplicated_adoptions}")
print(f"Conflicting duplicates: {conflicting_duplicated_adoptions}")

Exact duplicates: 8
Conflicting duplicates: 0


Here, eight duplicated adoption records were identified.  These were exact duplicates and removed, leaving us with 105 unique adoption ids:

In [790]:
adoptions=adoptions.drop_duplicates()

In [791]:
adoptions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 105 entries, 0 to 104
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   adoption_id              105 non-null    object 
 1   enquiry_id               105 non-null    object 
 2   animal_id                105 non-null    object 
 3   customer_id              105 non-null    object 
 4   adoption_date            105 non-null    object 
 5   fee_paid                 99 non-null     float64
 6   returned_within_30_days  98 non-null     object 
 7   return_date              15 non-null     object 
dtypes: float64(1), object(7)
memory usage: 7.4+ KB


***
**enquiry_id**

I first wanted to see if there were duplicated enquiry_ids in the adoptions DataFrame:

In [792]:
adoptions['enquiry_id'].duplicated().sum()

0

***
**animal_id**

I then wanted to see if there were duplicated animal_ids in the adoptions DataFrame:

In [793]:
adoptions['animal_id'].duplicated().sum()

0

In [794]:
adoptions.groupby('animal_id').size().value_counts()

1    105
Name: count, dtype: int64

We can be confident that there are no duplicated animal_ids in the adoptions DataFrame.

***
**customer_id**

Customer ids were also checked for duplicates however it was clear from further investigation that this is normal behaviour in that customers may want to adopt more than one pet.

In [795]:
adoptions['customer_id'].duplicated().sum()

15

In [796]:
duplicate_customer_adoptions = (
    adoptions[
    adoptions['customer_id'].duplicated(keep=False)
    ]
    .sort_values('customer_id')
)
duplicate_customer_adoptions.head(50)

,adoption_id,enquiry_id,animal_id,customer_id,adoption_date,fee_paid,returned_within_30_days,return_date
48,AD0049,E0292,A003,C0041,02 May 2025,75.0,No,NaN
41,AD0042,E0260,A153,C0041,27/01/2025,150.0,No,NaN
23,AD0024,E0234,A004,C0052,2025-05-03 00:00:00,NaN,No,NaN
38,AD0039,E0311,A038,C0052,07 Apr 2025,175.0,No,NaN
4,AD0005,E0283,A117,C0058,10/03/2025,200.0,No,NaN
43,AD0044,E0056,A125,C0058,24/01/2025,200.0,No,NaN
83,AD0084,E0037,A152,C0067,03-05-2025,200.0,No,NaN
31,AD0032,E0265,A109,C0067,2025/03/14,95.0,No,NaN
49,AD0050,E0244,A122,C0098,21/03/2025,95.0,NaN,NaN
13,AD0014,E0169,A010,C0098,02-05-2025,95.0,NaN,NaN


***
**adoption_date**

We had earlier ascertained that adoption_date was in the object format however if we wanted to perform any calculations on our data at a later stage, it would be preferable to format as datetime.  First, let's see what different dates we have to deal with:

In [797]:
adoptions['adoption_date'].astype(str).sort_values().unique()

array([' 02 February 2025 ', ' 03 Mar 2025 ', ' 13 Apr 2025 ',
       ' 14 Apr 2025 ', ' 14 March 2025 ', ' 16 Feb 2025 ',
       ' 2025-02-28 00:00:00 ', ' 2025-03-18 00:00:00 ',
       ' 2025-03-22 00:00:00 ', ' 2025-04-19 00:00:00 ', ' 2025/03/29 ',
       ' 27/01/2025 ', ' 30 January 2025 ', ' 30-01-2025 ',
       '01 April 2025', '01/05/2025', '02 May 2025', '02-05-2025',
       '02/03/2025', '03-05-2025', '05 Apr 2025', '06 February 2025',
       '06 May 2025', '06/02/2025', '06/04/2025', '07 Apr 2025',
       '07 Feb 2025', '07/02/2025', '09/04/2025', '09/05/2025',
       '10 April 2025', '10 Mar 2025', '10 March 2025', '10/02/2025',
       '10/03/2025', '10/05/2025', '13 May 2025', '14 Apr 2025',
       '16 Feb 2025', '16 March 2025', '16/03/2025', '17 Jan 2025',
       '18 Feb 2025', '18 March 2025', '18-04-2025', '18/02/2025',
       '19 Feb 2025', '19-02-2025', '20 February 2025', '20-04-2025',
       '2025-01-26 00:00:00', '2025-02-02 00:00:00',
       '2025-02-03 00:00:00'

In [798]:
adoptions['adoption_date'] = (
    adoptions['adoption_date']
    .astype(str)
    .str.strip()
)

In [799]:
adoptions['adoption_date'] = pd.to_datetime(
    adoptions['adoption_date'],
    errors='coerce',
    format='mixed',
    dayfirst=True
    )

In [800]:
adoptions['adoption_date'].astype(str).sort_values().unique()

array(['2025-01-17', '2025-01-20', '2025-01-24', '2025-01-26',
       '2025-01-27', '2025-01-30', '2025-01-31', '2025-02-02',
       '2025-02-03', '2025-02-05', '2025-02-06', '2025-02-07',
       '2025-02-09', '2025-02-10', '2025-02-16', '2025-02-18',
       '2025-02-19', '2025-02-20', '2025-02-24', '2025-02-27',
       '2025-02-28', '2025-03-02', '2025-03-03', '2025-03-08',
       '2025-03-09', '2025-03-10', '2025-03-14', '2025-03-16',
       '2025-03-18', '2025-03-21', '2025-03-22', '2025-03-24',
       '2025-03-26', '2025-03-27', '2025-03-29', '2025-03-30',
       '2025-03-31', '2025-04-01', '2025-04-03', '2025-04-05',
       '2025-04-06', '2025-04-07', '2025-04-09', '2025-04-10',
       '2025-04-13', '2025-04-14', '2025-04-15', '2025-04-18',
       '2025-04-19', '2025-04-20', '2025-04-22', '2025-04-23',
       '2025-04-24', '2025-04-27', '2025-04-28', '2025-04-30',
       '2025-05-01', '2025-05-02', '2025-05-03', '2025-05-06',
       '2025-05-09', '2025-05-10', '2025-05-13'], dtype

We can now see that the adoption_date column has successfully been converted to datetime:

In [801]:
adoptions['adoption_date'].isna().sum()

0

In [802]:
adoptions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 105 entries, 0 to 104
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   adoption_id              105 non-null    object        
 1   enquiry_id               105 non-null    object        
 2   animal_id                105 non-null    object        
 3   customer_id              105 non-null    object        
 4   adoption_date            105 non-null    datetime64[ns]
 5   fee_paid                 99 non-null     float64       
 6   returned_within_30_days  98 non-null     object        
 7   return_date              15 non-null     object        
dtypes: datetime64[ns](1), float64(1), object(6)
memory usage: 7.4+ KB


***
**fee_paid**

No causes for concern in the fee_paid column, with fees paid in the region of £75 - £250.

In [803]:
adoptions['fee_paid'].describe()

count     99.000000
mean     147.222222
std       58.082551
min       75.000000
25%       95.000000
50%      120.000000
75%      200.000000
max      250.000000
Name: fee_paid, dtype: float64

In [804]:
adoptions['fee_paid'].sort_values().head()

81    75.0
48    75.0
47    75.0
71    75.0
73    75.0
Name: fee_paid, dtype: float64

In [805]:
adoptions['fee_paid'].sort_values(ascending=False).head()

104    250.0
61     250.0
69     250.0
16     250.0
87     250.0
Name: fee_paid, dtype: float64

***
**returned_within_30_days**

Upon viewing the data in the returned_within_30_days column, we can see that a number of entries will require standardisation, which we can handle in our SQL cleaning step.

In [806]:
adoptions['returned_within_30_days'].value_counts()

returned_within_30_days
No       67
Yes      12
 No       9
no        5
 Yes      2
NO        2
YES       1
Name: count, dtype: int64

***
**return_date**

We first view the different date types that have been input into the return_date column, format to datetime and check to see if formatting has been applied correctly:

In [807]:
adoptions['return_date'].astype(str).sort_values().unique()

array([' 19 February 2025 ', ' 24 Feb 2025 ', '10 March 2025',
       '14-04-2025', '17-04-2025', '18-02-2025', '2025-02-22 00:00:00',
       '2025-03-01 00:00:00', '2025-03-21', '2025-03-26',
       '2025-05-11 00:00:00', '2025/04/27', '24-02-2025', '26-04-2025',
       '30 April 2025', 'nan'], dtype=object)

In [808]:
adoptions['return_date'] = (
    adoptions['return_date']
    .astype(str)
    .str.strip()
)

In [809]:
adoptions['return_date'] = pd.to_datetime(
    adoptions['return_date'],
    errors='coerce',
    format='mixed',
    dayfirst=True
    )

In [810]:
adoptions['return_date'].astype(str).sort_values().unique()

array(['2025-02-18', '2025-02-19', '2025-02-22', '2025-02-24',
       '2025-03-01', '2025-03-10', '2025-03-21', '2025-03-26',
       '2025-04-14', '2025-04-17', '2025-04-26', '2025-04-27',
       '2025-04-30', '2025-05-11', 'NaT'], dtype=object)

In [811]:
adoptions['return_date'].isna().sum()

90

In [812]:
adoptions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 105 entries, 0 to 104
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   adoption_id              105 non-null    object        
 1   enquiry_id               105 non-null    object        
 2   animal_id                105 non-null    object        
 3   customer_id              105 non-null    object        
 4   adoption_date            105 non-null    datetime64[ns]
 5   fee_paid                 99 non-null     float64       
 6   returned_within_30_days  98 non-null     object        
 7   return_date              15 non-null     datetime64[ns]
dtypes: datetime64[ns](2), float64(1), object(5)
memory usage: 7.4+ KB


***
## animals

In [813]:
animals.head()

,animal_id,name,species,breed,age_years,colour,sex,intake_date,health_status,adoptable
0,A001,Luna,Dog,Bulldog,4.0,Brown,M,2025/01/26,In Treatment,No
1,A002,Luna,Dog,Whippet,3.0,Cream,F,03/09/2024,Minor Issues,No
2,A003,Finn,Dog,Whippet,1.0,Tortoiseshell,M,24 Nov 2024,Minor Issues,No
3,A004,Nala,Cat,Russian Blue,8.0,Tortoiseshell,m,2025-02-01,Minor Issues,NaN
4,A005,Tilly,Cat,Siamese,3.0,Black,f,2024-08-04,NaN,Yes


In [814]:
animals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   animal_id      172 non-null    object 
 1   name           172 non-null    object 
 2   species        172 non-null    object 
 3   breed          172 non-null    object 
 4   age_years      156 non-null    float64
 5   colour         165 non-null    object 
 6   sex            167 non-null    object 
 7   intake_date    172 non-null    object 
 8   health_status  168 non-null    object 
 9   adoptable      159 non-null    object 
dtypes: float64(1), object(9)
memory usage: 13.6+ KB


In [815]:
animals.describe()

,age_years
count,156.000000
mean,4.948718
std,3.522819
min,0.000000
25%,2.000000
50%,4.000000
75%,7.250000
max,14.000000


***
**animal_id**

We being our process again by investigating and dealing with any duplicates in the animal_id column:

In [816]:
animals['animal_id'].duplicated().sum()

12

In [817]:
duplicate_animal_id = (
    animals[
    animals['animal_id'].duplicated(keep=False)
    ]
    .sort_values('animal_id')
)

In [818]:
exact_animal_ids = animals.duplicated().sum()

duplicate_animal_ids = (
    animals['animal_id'].duplicated().sum()
)
conflicting_animal_ids = (
    duplicate_animal_ids - exact_animal_ids
)
print(f"Exact duplicates: {exact_animal_ids}")
print(f"Conflicting duplicates: {conflicting_animal_ids}")

Exact duplicates: 12
Conflicting duplicates: 0


We see from the above that there are twelve duplicated records, so we will retain one instance of these:

In [819]:
animals=animals.drop_duplicates()

In [820]:
animals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 160 entries, 0 to 159
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   animal_id      160 non-null    object 
 1   name           160 non-null    object 
 2   species        160 non-null    object 
 3   breed          160 non-null    object 
 4   age_years      147 non-null    float64
 5   colour         153 non-null    object 
 6   sex            157 non-null    object 
 7   intake_date    160 non-null    object 
 8   health_status  156 non-null    object 
 9   adoptable      148 non-null    object 
dtypes: float64(1), object(9)
memory usage: 13.8+ KB


***
**name**

In [821]:
animals['name'].astype(str).sort_values().unique()

array(['Archie', 'Bailey', 'Bella', 'Biscuit', 'Bonnie', 'Buddy',
       'Charlie', 'Coco', 'Daisy', 'Finn', 'Jasper', 'Leo', 'Lola',
       'Lucy', 'Luna', 'Maisie', 'Max', 'Milo', 'Misty', 'Molly',
       'Murphy', 'Nala', 'Oscar', 'Pepper', 'Poppy', 'Rocky', 'Rosie',
       'Ruby', 'Sasha', 'Shadow', 'Simba', 'Teddy', 'Tilly', 'Willow',
       'Ziggy'], dtype=object)

***
**species**

In [822]:
animals['species'].astype(str).sort_values().unique()

array([' Cat ', ' Dog ', 'CAT', 'Cat', 'DOG', 'Dog', 'cat', 'dog'],
      dtype=object)

We will standardise Dog and Cat later in our SQL cleaning step.

***
**breed**

In [823]:
animals['breed'].astype(str).sort_values().unique()

array([' Beagle ', ' Collie ', ' Dachshund ', ' Labrador ', ' Persian ',
       'Beagle', 'Bengal', 'British Shorthair', 'Bulldog', 'Collie',
       'Dachshund', 'Domestic Longhair', 'Domestic Shorthair',
       'GERMAN SHEPHERD', 'German Shepherd', 'Greyhound', 'Husky',
       'LABRADOR', 'Labrador', 'Maine Coon', 'Mixed Breed',
       'Norwegian Forest Cat', 'PERSIAN', 'Persian', 'Poodle',
       'RUSSIAN BLUE', 'Ragdoll', 'Russian Blue', 'Siamese', 'Spaniel',
       'Sphynx', 'Staffy', 'Tabby', 'Terrier', 'Tuxedo', 'Whippet',
       'bulldog', 'dachshund', 'labrador', 'mixed breed', 'ragdoll',
       'siamese', 'tuxedo', 'whippet'], dtype=object)

We will standardise those breeds that require it, plus deal with trailing whitespace, in our late SQL cleaning step.

***
**age_years**

In [824]:
animals['age_years'].describe()

count    147.000000
mean       4.891156
std        3.544491
min        0.000000
25%        2.000000
50%        4.000000
75%        7.000000
max       14.000000
Name: age_years, dtype: float64

In [825]:
animals['age_years'].sort_values().dropna().tail()

142    13.0
100    13.0
14     13.0
134    14.0
66     14.0
Name: age_years, dtype: float64

In [826]:
animals.loc[
    animals["age_years"] == 0,
]

,animal_id,name,species,breed,age_years,colour,sex,intake_date,health_status,adoptable
20,A021,Tilly,Cat,Bengal,0.0,Ginger,M,17 Feb 2025,Minor Issues,no
47,A048,Archie,Cat,Siamese,0.0,Brindle,M,2025-02-19,good,Yes
53,A054,Max,Cat,Russian Blue,0.0,cream,F,2025-02-19 00:00:00,CHRONIC,Yes
58,A059,Lola,Dog,Greyhound,0.0,Calico,F,2025-01-08,Good,Yes
94,A095,Ruby,Cat,Maine Coon,0.0,NaN,M,13/12/2024,Good,NaN
110,A111,Simba,DOG,Dachshund,0.0,grey,M,03 February 2025,good,Yes
112,A113,Tilly,Cat,Domestic Shorthair,0.0,Black,M,2025/02/02,Good,Yes
117,A118,Poppy,dog,Husky,0.0,Black,M,2025-02-14,minor issues,Yes
135,A136,Jasper,Cat,Domestic Shorthair,0.0,Calico,F,2025-02-13 00:00:00,Good,No


The span of age_years runs from 0 to 14 however as it is not possible to adopt an animal age 0 years, we will set 0 to NULL in the SQL cleaning step.

***
**colour**

In [827]:
animals['colour'].value_counts()

colour
Black              17
Cream              16
Tortoiseshell      14
Ginger             11
Grey               10
Calico             10
Tabby              10
Brown               8
White               8
Black & White       8
Brindle             7
tabby               3
BLACK & WHITE       3
 Brown              2
GINGER              2
cream               2
grey                2
 Black              2
CALICO              2
white               2
 Brindle            2
black               2
ginger              1
 Calico             1
brown               1
black & white       1
 Ginger             1
 White              1
 Grey               1
 Black & White      1
calico              1
GREY                1
Name: count, dtype: int64

We will standardise those colours that require it, plus deal with trailing whitespace, in our late SQL cleaning step.

***
**sex**

In [828]:
animals['sex'].value_counts()

sex
F      70
M      66
f       9
 M      5
 F      4
m       3
Name: count, dtype: int64

We will standardise to M and F, plus deal with trailing whitespace, in our late SQL cleaning step.

***
**intake_date**

Again, as our dates were formatted as object, we will seek to re-format as datetime.

In [829]:
animals['intake_date'].astype(str).sort_values().unique()

array([' 01 January 2025 ', ' 03-02-2025 ', ' 04 Nov 2024 ',
       ' 05-10-2024 ', ' 14/08/2024 ', ' 17 Aug 2024 ',
       ' 19 October 2024 ', ' 2024-09-01 00:00:00 ',
       ' 2024-10-23 00:00:00 ', ' 2024-10-30 ', ' 2024-11-20 ',
       ' 2024/08/10 ', ' 2024/12/22 ', ' 2025-02-01 ',
       ' 2025-02-28 00:00:00 ', ' 2025/01/26 ', ' 2025/02/28 ',
       ' 24/01/2025 ', ' 28-12-2024 ', '01 Nov 2024', '01-02-2025',
       '02-02-2025', '03 February 2025', '03 October 2024', '03-11-2024',
       '03/09/2024', '04 Feb 2025', '04 Jan 2025', '04/01/2025',
       '05 February 2025', '05-11-2024', '06 Aug 2024',
       '06 February 2025', '06 November 2024', '06 October 2024',
       '06 Sep 2024', '06-11-2024', '06/08/2024', '06/12/2024',
       '07 December 2024', '07 February 2025', '07 Oct 2024',
       '07/09/2024', '07/11/2024', '08 August 2024', '08 December 2024',
       '09 Dec 2024', '09/12/2024', '10 Dec 2024', '10 January 2025',
       '10-01-2025', '10-10-2024', '10/01/2025', 

In [830]:
animals['intake_date'] = (
    animals['intake_date']
    .astype(str)
    .str.strip()
)

In [831]:
animals['intake_date'] = pd.to_datetime(
    animals['intake_date'],
    errors='coerce',
    format='mixed',
    dayfirst=True
    )

In [832]:
animals['intake_date'].astype(str).sort_values().unique()

array(['2024-08-04', '2024-08-06', '2024-08-07', '2024-08-08',
       '2024-08-10', '2024-08-11', '2024-08-13', '2024-08-14',
       '2024-08-15', '2024-08-17', '2024-08-18', '2024-08-19',
       '2024-08-20', '2024-08-26', '2024-08-27', '2024-08-28',
       '2024-08-30', '2024-09-01', '2024-09-03', '2024-09-06',
       '2024-09-07', '2024-09-09', '2024-09-10', '2024-09-11',
       '2024-09-12', '2024-09-14', '2024-09-16', '2024-09-17',
       '2024-09-19', '2024-09-20', '2024-09-21', '2024-09-22',
       '2024-09-26', '2024-09-28', '2024-09-30', '2024-10-03',
       '2024-10-04', '2024-10-05', '2024-10-06', '2024-10-07',
       '2024-10-10', '2024-10-11', '2024-10-16', '2024-10-17',
       '2024-10-18', '2024-10-19', '2024-10-20', '2024-10-22',
       '2024-10-23', '2024-10-26', '2024-10-27', '2024-10-28',
       '2024-10-30', '2024-11-01', '2024-11-03', '2024-11-04',
       '2024-11-05', '2024-11-06', '2024-11-07', '2024-11-10',
       '2024-11-12', '2024-11-13', '2024-11-19', '2024-

In [833]:
animals['intake_date'].isna().sum()

0

In [834]:
animals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 160 entries, 0 to 159
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   animal_id      160 non-null    object        
 1   name           160 non-null    object        
 2   species        160 non-null    object        
 3   breed          160 non-null    object        
 4   age_years      147 non-null    float64       
 5   colour         153 non-null    object        
 6   sex            157 non-null    object        
 7   intake_date    160 non-null    datetime64[ns]
 8   health_status  156 non-null    object        
 9   adoptable      148 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(8)
memory usage: 13.8+ KB


***
**health_status**

In [835]:
animals['health_status'].value_counts()

health_status
Good              79
Minor Issues      24
In Treatment      16
good               8
GOOD               7
Chronic            5
minor issues       4
CHRONIC            3
 Minor Issues      2
 Good              2
IN TREATMENT       2
MINOR ISSUES       2
 In Treatment      1
in treatment       1
Name: count, dtype: int64

We will standardise, plus deal with trailing whitespace, in our late SQL cleaning step.

***
**adoptable**

In [836]:
animals['adoptable'].value_counts()

adoptable
Yes      90
No       38
yes       6
 Yes      6
no        4
YES       3
NO        1
Name: count, dtype: int64

In [837]:
animals.groupby('adoptable')['health_status'].value_counts(dropna=False)

adoptable  health_status 
 Yes       Good               3
           CHRONIC            1
           IN TREATMENT       1
           minor issues       1
NO         Minor Issues       1
No         Good              12
           Minor Issues       9
           In Treatment       7
           good               3
           Chronic            2
           GOOD               2
           NaN                2
           in treatment       1
YES        GOOD               1
           Good               1
           In Treatment       1
Yes        Good              55
           Minor Issues       7
           In Treatment       6
           Chronic            3
           GOOD               3
           good               3
           minor issues       3
           CHRONIC            2
           MINOR ISSUES       2
           NaN                2
            Good              1
            In Treatment      1
            Minor Issues      1
           IN TREATMENT       1
no         Min

***
## enquiries

In [838]:
enquiries.head()

,enquiry_id,animal_id,customer_id,enquiry_date,channel,response_time_hours,enquiry_status
0,E0001,A115,C0369,12 Apr 2025,Walk-in,8.0,Closed
1,E0002,A121,C0296,29/03/2025,Facebook,10.0,Open
2,E0003,A095,C0288,2025-03-17,Website,26.0,Closed
3,E0004,A030,C0122,13-02-2025,Email,34.0,Closed
4,E0005,A011,C0149,22-03-2025,Website,7.0,Closed


In [839]:
enquiries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 332 entries, 0 to 331
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   enquiry_id           332 non-null    object 
 1   animal_id            332 non-null    object 
 2   customer_id          332 non-null    object 
 3   enquiry_date         332 non-null    object 
 4   channel              323 non-null    object 
 5   response_time_hours  309 non-null    float64
 6   enquiry_status       332 non-null    object 
dtypes: float64(1), object(6)
memory usage: 18.3+ KB


In [840]:
enquiries.describe()

,response_time_hours
count,309.000000
mean,18.436893
std,25.963262
min,0.000000
25%,5.000000
50%,10.000000
75%,19.000000
max,120.000000


***
**enquiry_id**

In [841]:
enquiries['enquiry_id'].duplicated().sum()

12

Firstly, we see there being 12 duplicated entries in the 'enquiry_id' column which will need investigating.

In [842]:
duplicate_enquiries = (
    enquiries[
    enquiries['enquiry_id'].duplicated(keep=False)
    ]
    .sort_values('enquiry_id')
)

In [843]:
exact_duplicates = enquiries.duplicated().sum()

duplicate_enquiry_rows = (
    enquiries['enquiry_id'].duplicated().sum()
)
conflicting_duplicates = (
    duplicate_enquiry_rows - exact_duplicates
)
print(f"Exact duplicates: {exact_duplicates}")
print(f"Conflicting duplicates: {conflicting_duplicates}")

Exact duplicates: 12
Conflicting duplicates: 0


Here, twelve duplicated enquiry records were identified.  These were exact duplicates and removed, leaving us with 320 unique enquiry ids:

In [844]:
enquiries=enquiries.drop_duplicates()

In [845]:
enquiries.info()

<class 'pandas.core.frame.DataFrame'>
Index: 320 entries, 0 to 319
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   enquiry_id           320 non-null    object 
 1   animal_id            320 non-null    object 
 2   customer_id          320 non-null    object 
 3   enquiry_date         320 non-null    object 
 4   channel              311 non-null    object 
 5   response_time_hours  299 non-null    float64
 6   enquiry_status       320 non-null    object 
dtypes: float64(1), object(6)
memory usage: 20.0+ KB


***
**customer_id**

I then did an initial check on customer_id to see if there were any duplicate records but it looks as though customers have genuinely made more than one enquiry, sometimes on different channels (i.e., phone, web, email), and for more than one animal

In [846]:
enquiries['customer_id'].duplicated().sum()

99

In [847]:
duplicate_customers = (
    enquiries[
    enquiries['customer_id'].duplicated(keep=False)
    ]
    .sort_values('customer_id')
)
duplicate_customers

,enquiry_id,animal_id,customer_id,enquiry_date,channel,response_time_hours,enquiry_status
110,E0111,A007,C0001,16-01-2025,Website,7.0,Closed
242,E0243,A092,C0001,27 Jan 2025,Website,14.0,Closed
52,E0053,A122,C0003,2025/02/24,Email,37.0,Closed
250,E0251,A124,C0003,05-04-2025,Phone,5.0,Open
44,E0045,A155,C0003,27/03/2025,Website,23.0,Withdrawn
...,...,...,...,...,...,...,...
0,E0001,A115,C0369,12 Apr 2025,Walk-in,8.0,Closed
27,E0028,A016,C0375,27/01/2025,Website,12.0,Closed
295,E0296,A112,C0375,31-03-2025,Facebook,13.0,Withdrawn
97,E0098,A092,C0377,15-02-2025,Phone,7.0,CLOSED


In [848]:
exact_duplicated_customers = enquiries.duplicated().sum()

duplicate_customers_rows = (
    enquiries['customer_id'].duplicated().sum()
)
conflicting_duplicated_customers = (
    duplicate_customers_rows - exact_duplicated_customers
)
print(f"Exact duplicates: {exact_duplicated_customers}")
print(f"Conflicting duplicates: {conflicting_duplicated_customers}")

Exact duplicates: 0
Conflicting duplicates: 99


***
**enquiry_date**

Dates appear in different formats so we convert to datetime and check that formatting has been corrected:

In [849]:
enquiries['enquiry_date'].astype(str).sort_values().unique()

array([' 01 Feb 2025 ', ' 03 Mar 2025 ', ' 03/02/2025 ', ' 05 Apr 2025 ',
       ' 05/02/2025 ', ' 06-04-2025 ', ' 07 Jan 2025 ', ' 10-01-2025 ',
       ' 10-04-2025 ', ' 11-01-2025 ', ' 12 Apr 2025 ', ' 15/04/2025 ',
       ' 16 Feb 2025 ', ' 16 Mar 2025 ', ' 17 Feb 2025 ', ' 20/02/2025 ',
       ' 2025-01-08 ', ' 2025-01-15 ', ' 2025-01-24 ',
       ' 2025-01-26 00:00:00 ', ' 2025-01-31 00:00:00 ', ' 2025-02-01 ',
       ' 2025-02-19 00:00:00 ', ' 2025-02-26 ', ' 2025-03-03 00:00:00 ',
       ' 2025-03-04 00:00:00 ', ' 2025-03-07 00:00:00 ',
       ' 2025-03-14 00:00:00 ', ' 2025-03-15 ', ' 2025-03-17 ',
       ' 2025-03-30 00:00:00 ', ' 2025-03-31 00:00:00 ', ' 2025-04-02 ',
       ' 2025-04-03 ', ' 2025/01/18 ', ' 2025/03/15 ', ' 21 Mar 2025 ',
       ' 22/03/2025 ', ' 23 Jan 2025 ', ' 25 Feb 2025 ',
       ' 26 February 2025 ', ' 26/02/2025 ', ' 31 March 2025 ',
       '01 Apr 2025', '01 February 2025', '01-02-2025', '01/04/2025',
       '02 Feb 2025', '02 Mar 2025', '02/04/2025',

In [850]:
enquiries['enquiry_date'] = (
    enquiries['enquiry_date']
    .astype(str)
    .str.strip()
)

In [851]:
enquiries['enquiry_date'] = pd.to_datetime(
    enquiries['enquiry_date'],
    errors='coerce',
    format='mixed',
    dayfirst=True

    )

In [852]:
enquiries['enquiry_date'].astype(str).sort_values().unique()

array(['2025-01-05', '2025-01-06', '2025-01-07', '2025-01-08',
       '2025-01-09', '2025-01-10', '2025-01-11', '2025-01-12',
       '2025-01-13', '2025-01-14', '2025-01-15', '2025-01-16',
       '2025-01-17', '2025-01-18', '2025-01-19', '2025-01-20',
       '2025-01-21', '2025-01-22', '2025-01-23', '2025-01-24',
       '2025-01-25', '2025-01-26', '2025-01-27', '2025-01-28',
       '2025-01-29', '2025-01-30', '2025-01-31', '2025-02-01',
       '2025-02-02', '2025-02-03', '2025-02-04', '2025-02-05',
       '2025-02-06', '2025-02-07', '2025-02-08', '2025-02-09',
       '2025-02-10', '2025-02-11', '2025-02-12', '2025-02-13',
       '2025-02-15', '2025-02-16', '2025-02-17', '2025-02-18',
       '2025-02-19', '2025-02-20', '2025-02-22', '2025-02-23',
       '2025-02-24', '2025-02-25', '2025-02-26', '2025-02-27',
       '2025-02-28', '2025-03-02', '2025-03-03', '2025-03-04',
       '2025-03-05', '2025-03-06', '2025-03-07', '2025-03-08',
       '2025-03-09', '2025-03-11', '2025-03-12', '2025-

In [853]:
enquiries['enquiry_date'].isna().sum()

0

In [854]:
enquiries.info()

<class 'pandas.core.frame.DataFrame'>
Index: 320 entries, 0 to 319
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   enquiry_id           320 non-null    object        
 1   animal_id            320 non-null    object        
 2   customer_id          320 non-null    object        
 3   enquiry_date         320 non-null    datetime64[ns]
 4   channel              311 non-null    object        
 5   response_time_hours  299 non-null    float64       
 6   enquiry_status       320 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 20.0+ KB


***
**channel**

In [855]:
enquiries['channel'].value_counts()

channel
Website       138
Phone          49
Email          34
Facebook       26
Walk-in        14
website        13
phone           8
 Website        7
WEBSITE         5
email           4
 Email          3
facebook        2
walk-in         2
 Facebook       2
 Walk-in        1
 Phone          1
PHONE           1
WALK-IN         1
Name: count, dtype: int64

Upon viewing the data in the channel column, we can see that a number of entries will require standardisation, which we can handle in our SQL cleaning step.

***
**response_time_hours**

Upon initial analysis of our response_time_hours column, we can see the range of response times, from 0 to 97 hours:

In [856]:
enquiries['response_time_hours'].sort_values().head()

318    0.0
37     0.0
268    0.0
46     0.0
258    0.0
Name: response_time_hours, dtype: float64

In [857]:
enquiries['response_time_hours'].sort_values(ascending=False).head()

273    120.0
24     118.0
32     114.0
134    112.0
316    111.0
Name: response_time_hours, dtype: float64

We shall nullify those responses of 0 hours in our later SQL cleaning stage, as a response time of zero would not be possible.

***
**enquiry_status**

In [858]:
enquiries['enquiry_status'].astype(str).sort_values().unique()

array([' Closed ', ' Open ', 'CLOSED', 'Closed', 'OPEN', 'Open',
       'WITHDRAWN', 'Withdrawn', 'closed', 'open', 'withdrawn'],
      dtype=object)

Upon viewing the data in the enquiry_status column, we can see that a number of entries will require standardisation, which we can handle in our SQL cleaning step.

***
## Visits

In [859]:
visits.head()

,visit_id,enquiry_id,visit_date,showed_up,visit_type
0,V0001,E0130,27 Mar 2025,No,In-person
1,V0002,E0023,15 Feb 2025,Yes,Virtual
2,V0003,E0023,2025/02/20,No,Home Check
3,V0004,E0072,2025-01-25,Yes,Home Check
4,V0005,E0311,2025-04-04,Yes,Home Check


In [860]:
visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   visit_id    372 non-null    object
 1   enquiry_id  372 non-null    object
 2   visit_date  372 non-null    object
 3   showed_up   345 non-null    object
 4   visit_type  372 non-null    object
dtypes: object(5)
memory usage: 14.7+ KB


***
**visit_id**

First, I will check to see if visit_ids are individual, i.e., that we have no duplicated entries:

In [861]:
visits['visit_id'].duplicated().sum()

10

In [862]:
duplicate_visits = (
    visits[
    visits['visit_id'].duplicated(keep=False)
    ]
    .sort_values('visit_id')
)

In [863]:
exact_duplicated_visits = visits.duplicated().sum()

duplicate_visits_rows = (
    visits['visit_id'].duplicated().sum()
)
conflicting_duplicated_visits = (
    duplicate_visits_rows - exact_duplicated_visits
)
print(f"Exact duplicates: {exact_duplicated_visits}")
print(f"Conflicting duplicates: {conflicting_duplicated_visits}")

Exact duplicates: 10
Conflicting duplicates: 0


Here, ten duplicated enquiry records were identified.  These were exact duplicates and removed, leaving us with 362 unique visit ids:

In [864]:
visits=visits.drop_duplicates()

In [865]:
visits.info()

<class 'pandas.core.frame.DataFrame'>
Index: 362 entries, 0 to 361
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   visit_id    362 non-null    object
 1   enquiry_id  362 non-null    object
 2   visit_date  362 non-null    object
 3   showed_up   335 non-null    object
 4   visit_type  362 non-null    object
dtypes: object(5)
memory usage: 17.0+ KB


***
**enquiry_id**

I also wanted to investigate whether there were any duplicated enquiry_ids within the visits DataFrame, and it transpires there were 92.

In [866]:
visits['enquiry_id'].duplicated().sum()

92

In [867]:
duplicate_enquiry_ids = (
    visits[
    visits['enquiry_id'].duplicated(keep=False)
    ]
    .sort_values('enquiry_id')
)
duplicate_enquiry_ids

,visit_id,enquiry_id,visit_date,showed_up,visit_type
262,V0263,E0002,2025-04-05,Yes,Virtual
263,V0264,E0002,2025-04-13 00:00:00,YES,VIRTUAL
264,V0265,E0002,09-04-2025,Yes,In-person
73,V0074,E0011,27/02/2025,No,In-person
72,V0073,E0011,25 February 2025,Yes,Virtual
...,...,...,...,...,...
114,V0115,E0313,03 February 2025,Yes,In-person
115,V0116,E0313,2025-02-06,No,IN-PERSON
116,V0117,E0313,13/02/2025,yes,HOME CHECK
110,V0111,E0315,2025/03/12,Yes,Virtual


In [868]:
exact_duplicated_enquiries = visits.duplicated().sum()

duplicate_enquiries_rows = (
    visits['enquiry_id'].duplicated().sum()
)
conflicting_duplicated_enquiries = (
    duplicate_enquiries_rows - exact_duplicated_enquiries
)
print(f"Exact duplicates: {exact_duplicated_enquiries}")
print(f"Conflicting duplicates: {conflicting_duplicated_enquiries}")

Exact duplicates: 0
Conflicting duplicates: 92


From an initial overview of the data, these appear to be times when an enquiry genuinely led to more than one visit but others.

In [869]:
visits.groupby(
    ['enquiry_id', 'visit_date']
).size().reset_index(name='count')

,enquiry_id,visit_date,count
0,E0001,26/04/2025,1
1,E0002,09-04-2025,1
2,E0002,2025-04-05,1
3,E0002,2025-04-13 00:00:00,1
4,E0004,2025-02-14,1
...,...,...,...
357,E0315,14-03-2025,1
358,E0316,2025/04/17,1
359,E0317,27 Mar 2025,1
360,E0318,24/03/2025,1


***
**visit_date**

Similar to the enquiry_date column in the enquiries DataFrame, the data type was formatted as object.  In case we wish to run calculations on this column in future, we will format to datetime.

In [870]:
visits['visit_date'].astype(str).sort_values().unique()

array([' 03-03-2025 ', ' 05-03-2025 ', ' 05/02/2025 ', ' 06-02-2025 ',
       ' 08 February 2025 ', ' 09 Apr 2025 ', ' 10 March 2025 ',
       ' 11 February 2025 ', ' 14/02/2025 ', ' 15 April 2025 ',
       ' 15/04/2025 ', ' 17 Apr 2025 ', ' 17 Mar 2025 ', ' 17-01-2025 ',
       ' 18 April 2025 ', ' 18 Jan 2025 ', ' 19 February 2025 ',
       ' 2025-01-25 ', ' 2025-01-27 00:00:00 ', ' 2025-02-13 00:00:00 ',
       ' 2025-02-19 00:00:00 ', ' 2025-02-28 ', ' 2025-03-08 00:00:00 ',
       ' 2025-03-27 ', ' 2025-04-12 00:00:00 ', ' 2025-04-13 00:00:00 ',
       ' 2025-04-14 00:00:00 ', ' 2025/03/08 ', ' 2025/03/10 ',
       ' 2025/03/12 ', ' 2025/03/25 ', ' 22/01/2025 ', ' 22/02/2025 ',
       ' 22/03/2025 ', ' 23/03/2025 ', ' 27 Mar 2025 ', ' 28 Jan 2025 ',
       ' 29/01/2025 ', ' 29/03/2025 ', '01 Apr 2025', '01 April 2025',
       '01 Feb 2025', '01-04-2025', '01/05/2025', '02-03-2025',
       '02/02/2025', '02/03/2025', '02/04/2025', '03 Apr 2025',
       '03 April 2025', '03 February

In [871]:
visits['visit_date'] = (
    visits['visit_date']
    .astype(str)
    .str.strip()
)

In [872]:
visits['visit_date'] = pd.to_datetime(
    visits['visit_date'],
    errors='coerce',
    format='mixed',
    dayfirst=True
    )

In [873]:
visits['visit_date'].astype(str).sort_values().unique()

array(['2025-01-10', '2025-01-11', '2025-01-13', '2025-01-14',
       '2025-01-15', '2025-01-16', '2025-01-17', '2025-01-18',
       '2025-01-19', '2025-01-20', '2025-01-21', '2025-01-22',
       '2025-01-23', '2025-01-24', '2025-01-25', '2025-01-26',
       '2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30',
       '2025-01-31', '2025-02-01', '2025-02-02', '2025-02-03',
       '2025-02-04', '2025-02-05', '2025-02-06', '2025-02-07',
       '2025-02-08', '2025-02-09', '2025-02-10', '2025-02-11',
       '2025-02-12', '2025-02-13', '2025-02-14', '2025-02-15',
       '2025-02-16', '2025-02-17', '2025-02-18', '2025-02-19',
       '2025-02-20', '2025-02-22', '2025-02-23', '2025-02-24',
       '2025-02-25', '2025-02-26', '2025-02-27', '2025-02-28',
       '2025-03-01', '2025-03-02', '2025-03-03', '2025-03-04',
       '2025-03-05', '2025-03-06', '2025-03-07', '2025-03-08',
       '2025-03-09', '2025-03-10', '2025-03-11', '2025-03-12',
       '2025-03-13', '2025-03-14', '2025-03-15', '2025-

In [874]:
visits['visit_date'].isna().sum()

0

In [875]:
visits.info()

<class 'pandas.core.frame.DataFrame'>
Index: 362 entries, 0 to 361
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   visit_id    362 non-null    object        
 1   enquiry_id  362 non-null    object        
 2   visit_date  362 non-null    datetime64[ns]
 3   showed_up   335 non-null    object        
 4   visit_type  362 non-null    object        
dtypes: datetime64[ns](1), object(4)
memory usage: 17.0+ KB


***
**showed_up**

In [876]:
visits['showed_up'].value_counts()

showed_up
Yes      194
No        68
yes       24
YES       22
 Yes      16
NO         5
no         3
 No        3
Name: count, dtype: int64

Upon viewing the data in the showed_up column, we can see that a number of entries will require standardisation, which we can handle in our SQL cleaning step.

***
**visit_type**

In [877]:
visits['visit_type'].value_counts()

visit_type
In-person       105
Home Check       99
Virtual          92
virtual          12
VIRTUAL           9
 Home Check       8
IN-PERSON         8
in-person         7
 Virtual          7
home check        6
 In-person        6
HOME CHECK        3
Name: count, dtype: int64

Again, we can see that a number of entries will require standardisation, which we can handle in our SQL cleaning step.

***
## Create SQLite database

In [878]:
import sqlite3

conn = sqlite3.connect(
    "animal_shelter.db"
)

***
## Load tables into SQL

In [879]:
adoptions.to_sql(
   "adoptions",
   conn,
   if_exists="replace",
   index=False
)

animals.to_sql(
   "animals",
   conn,
   if_exists="replace",
   index=False
)

enquiries.to_sql(
   "enquiries",
   conn,
   if_exists="replace",
   index=False
)

visits.to_sql(
   "visits",
   conn,
   if_exists="replace",
   index=False
)



362

***
## Cleaning log

**adoptions**

 - Standardised returned within 30 days inputs

**animals**

 - Standardised species inputs
 - Standardised breed inputs
 - Nullified animals with age of zero
 - Standardised colour inputs
 - Standardised sex inputs
 - Standardised health status inputs
 - Standardised adoptable status inputs

**enquiries**

 - Standardised channel inputs
 - Nullified response times with zero hours
 - Standardised enquiry status inputs

**visits**

 - Standardised showed up inputs
 - Standardised visit type inputs

***
## Cleaning data using SQL

***
**adoptions**

In [880]:
adoptions_query="""
SELECT
	adoption_id,
    enquiry_id,
	animal_id,
	customer_id,
	adoption_date,
	fee_paid,
	CASE
        WHEN returned_within_30_days IS NULL THEN 'Unknown'
		WHEN LOWER(TRIM(returned_within_30_days)) IN ('no')
			THEN 'No'
		WHEN LOWER(TRIM(returned_within_30_days)) IN ('yes')
			THEN 'Yes'
		ELSE TRIM(returned_within_30_days)
	END AS returned_within_30_days, 
	return_date
FROM adoptions
"""
cleaned_adoptions=pd.read_sql(adoptions_query, conn)
cleaned_adoptions

,adoption_id,enquiry_id,animal_id,customer_id,adoption_date,fee_paid,returned_within_30_days,return_date
0,AD0001,E0175,A136,C0161,2025-04-07 00:00:00,120.0,No,None
1,AD0002,E0274,A142,C0342,2025-01-24 00:00:00,75.0,No,None
2,AD0003,E0062,A022,C0263,2025-02-06 00:00:00,150.0,No,None
3,AD0004,E0097,A086,C0118,2025-04-23 00:00:00,100.0,Yes,2025-04-30 00:00:00
4,AD0005,E0283,A117,C0058,2025-03-10 00:00:00,200.0,No,None
...,...,...,...,...,...,...,...,...
100,AD0101,E0190,A118,C0349,2025-03-29 00:00:00,200.0,No,None
101,AD0102,E0042,A123,C0370,2025-04-24 00:00:00,NaN,No,None
102,AD0103,E0188,A135,C0107,2025-03-18 00:00:00,95.0,Yes,2025-03-21 00:00:00
103,AD0104,E0253,A100,C0208,2025-02-10 00:00:00,95.0,No,None


***
**animals**

In [881]:
animals_query="""
SELECT
	animal_id,
	name,
	CASE
		WHEN LOWER(TRIM(species)) IN ('cat')
			THEN 'Cat'
		WHEN LOWER(TRIM(species)) IN ('dog')
			THEN 'Dog'
		ELSE TRIM(species)
	END AS species,
	CASE
		WHEN LOWER(TRIM(breed)) IN ('german shepherd')
			THEN 'German Shepherd'
		WHEN LOWER(TRIM(breed)) IN ('labrador')
			THEN 'Labrador'
		WHEN LOWER(TRIM(breed)) IN ('persian')
			THEN 'Persian'
		WHEN LOWER(TRIM(breed)) IN ('russian blue')
			THEN 'Russian Blue'
		WHEN LOWER(TRIM(breed)) IN ('bulldog')
			THEN 'Bulldog'
		WHEN LOWER(TRIM(breed)) IN ('dachshund')
			THEN 'Dachshund'
		WHEN LOWER(TRIM(breed)) IN ('mixed breed')
			THEN 'Mixed Breed'
		WHEN LOWER(TRIM(breed)) IN ('ragdoll')
			THEN 'Ragdoll'
		WHEN LOWER(TRIM(breed)) IN ('siamese')
			THEN 'Siamese'
		WHEN LOWER(TRIM(breed)) IN ('tuxedo')
			THEN 'Tuxedo'
		WHEN LOWER(TRIM(breed)) IN ('whippet')
			THEN 'Whippet'
		ELSE TRIM(breed)
	END AS breed,
	CASE
		WHEN age_years == 0 THEN NULL
		ELSE age_years
	END AS age_years,
	CASE 
        WHEN colour IS NULL THEN 'Unknown'
		WHEN LOWER(TRIM(colour)) IN ('black & white')
			THEN 'Black & White'
		WHEN LOWER(TRIM(colour)) IN ('ginger')
			THEN 'Ginger'
		WHEN LOWER(TRIM(colour)) IN ('calico')
			THEN 'Calico'
		WHEN LOWER(TRIM(colour)) IN ('grey')
			THEN 'Grey'
        WHEN LOWER(TRIM(colour)) IN ('cream')
			THEN 'Cream'    
        WHEN LOWER(TRIM(colour)) IN ('brown')
			THEN 'Brown'
        WHEN LOWER(TRIM(colour)) IN ('black')
			THEN 'Black'
        WHEN LOWER(TRIM(colour)) IN ('white')
			THEN 'White'
        WHEN LOWER(TRIM(colour)) IN ('tabby')
			THEN 'Tabby'
		ELSE TRIM(colour)
	END AS colour, 
	CASE
        WHEN sex IS NULL THEN 'Unknown'
		WHEN LOWER(TRIM(sex)) IN ('f')
			THEN 'F'
		WHEN LOWER(TRIM(sex)) IN ('m')
			THEN 'M'
		ELSE TRIM (sex)
	END AS sex, 
	intake_date,
	CASE
        WHEN health_status IS NULL THEN 'Unknown'
		WHEN LOWER(TRIM(health_status)) IN ('good')
			THEN 'Good'
		WHEN LOWER(TRIM(health_status)) IN ('chronic')	
			THEN 'Chronic'
		WHEN LOWER(TRIM(health_status)) IN ('in treatment')
			THEN 'In Treatment'
		WHEN LOWER(TRIM(health_status)) IN ('minor issues')
			THEN 'Minor Issues'
		ELSE TRIM(health_status)
	END AS health_status, 
	CASE
		WHEN LOWER(TRIM(adoptable)) IN ('yes')
			THEN 'Yes'
		WHEN LOWER(TRIM(adoptable)) IN ('no')	
			THEN 'No'
        WHEN adoptable IS NULL OR LOWER(TRIM(adoptable)) = 'none' THEN 'Unknown'
		ELSE TRIM(adoptable)
	END AS adoptable
FROM animals
"""
cleaned_animals=pd.read_sql(animals_query, conn)
cleaned_animals


,animal_id,name,species,breed,age_years,colour,sex,intake_date,health_status,adoptable
0,A001,Luna,Dog,Bulldog,4.0,Brown,M,2025-01-26 00:00:00,In Treatment,No
1,A002,Luna,Dog,Whippet,3.0,Cream,F,2024-09-03 00:00:00,Minor Issues,No
2,A003,Finn,Dog,Whippet,1.0,Tortoiseshell,M,2024-11-24 00:00:00,Minor Issues,No
3,A004,Nala,Cat,Russian Blue,8.0,Tortoiseshell,M,2025-02-01 00:00:00,Minor Issues,Unknown
4,A005,Tilly,Cat,Siamese,3.0,Black,F,2024-08-04 00:00:00,Unknown,Yes
...,...,...,...,...,...,...,...,...,...,...
155,A156,Bailey,Cat,Sphynx,5.0,Black & White,F,2024-09-14 00:00:00,Chronic,Yes
156,A157,Milo,Dog,Poodle,NaN,Black,F,2024-08-06 00:00:00,Good,Yes
157,A158,Tilly,Dog,Beagle,12.0,Grey,F,2025-01-19 00:00:00,Good,Unknown
158,A159,Pepper,Cat,Bengal,1.0,Black,F,2024-12-10 00:00:00,Good,Unknown


***
**enquiries**

In [882]:
enquiries_query="""
SELECT
	enquiry_id,
	animal_id,
	customer_id,
	enquiry_date,
	CASE
        WHEN channel IS NULL THEN 'Unknown'
		WHEN LOWER(TRIM(channel)) IN ('website')
			THEN 'Website'
		WHEN LOWER(TRIM(channel)) IN ('facebook')
			THEN 'Facebook'
		WHEN LOWER(TRIM(channel)) IN ('walk-in')
			THEN 'Walk-In'
		WHEN LOWER(TRIM(channel)) IN ('phone')
			THEN 'Phone'
		WHEN LOWER(TRIM(channel)) IN ('email')
			THEN 'Email'
		ELSE TRIM (channel)
	END AS channel,
	CASE
        WHEN response_time_hours == 0 THEN NULL
        ELSE response_time_hours
    END AS response_time_hours,
    CASE
	WHEN LOWER(TRIM(enquiry_status)) IN ('closed')
		THEN 'Closed'
	WHEN LOWER(TRIM(enquiry_status)) IN ('open')
		THEN 'Open'
	WHEN LOWER(TRIM(enquiry_status)) IN ('withdrawn')
		THEN 'Withdrawn'
	ELSE TRIM(enquiry_status)
END AS enquiry_status
FROM enquiries
"""
cleaned_enquiries=pd.read_sql(enquiries_query, conn)
cleaned_enquiries


,enquiry_id,animal_id,customer_id,enquiry_date,channel,response_time_hours,enquiry_status
0,E0001,A115,C0369,2025-04-12 00:00:00,Walk-In,8.0,Closed
1,E0002,A121,C0296,2025-03-29 00:00:00,Facebook,10.0,Open
2,E0003,A095,C0288,2025-03-17 00:00:00,Website,26.0,Closed
3,E0004,A030,C0122,2025-02-13 00:00:00,Email,34.0,Closed
4,E0005,A011,C0149,2025-03-22 00:00:00,Website,7.0,Closed
...,...,...,...,...,...,...,...
315,E0316,A020,C0166,2025-04-09 00:00:00,Website,6.0,Closed
316,E0317,A011,C0361,2025-03-12 00:00:00,Facebook,111.0,Closed
317,E0318,A132,C0190,2025-03-16 00:00:00,Email,38.0,Closed
318,E0319,A008,C0024,2025-04-06 00:00:00,Website,NaN,Open


***
**visits**

In [883]:
visits_query="""
SELECT
	visit_id,
	enquiry_id,
	visit_date,
	CASE
        WHEN showed_up IS NULL THEN 'Unknown'
		WHEN LOWER(TRIM(showed_up)) IN ('yes')
			THEN 'Yes'
		WHEN LOWER(TRIM(showed_up)) IN ('no')
			THEN 'No'
		ELSE TRIM(showed_up)
	END AS showed_up,
	CASE
        WHEN visit_type IS NULL THEN 'Unknown'
		WHEN LOWER(TRIM(visit_type)) IN ('in-person')
			THEN 'In-Person'
		WHEN LOWER(TRIM(visit_type)) IN ('virtual')
			THEN 'Virtual'
		WHEN LOWER(TRIM(visit_type)) IN ('home check')
			THEN 'Home Check'
		ELSE TRIM(visit_type)
	END AS visit_type
FROM visits
"""
cleaned_visits=pd.read_sql(visits_query, conn)
cleaned_visits


,visit_id,enquiry_id,visit_date,showed_up,visit_type
0,V0001,E0130,2025-03-27 00:00:00,No,In-Person
1,V0002,E0023,2025-02-15 00:00:00,Yes,Virtual
2,V0003,E0023,2025-02-20 00:00:00,No,Home Check
3,V0004,E0072,2025-01-25 00:00:00,Yes,Home Check
4,V0005,E0311,2025-04-04 00:00:00,Yes,Home Check
...,...,...,...,...,...
357,V0358,E0158,2025-03-10 00:00:00,Yes,Virtual
358,V0359,E0187,2025-04-18 00:00:00,No,Home Check
359,V0360,E0187,2025-04-22 00:00:00,No,In-Person
360,V0361,E0239,2025-01-27 00:00:00,Yes,Home Check


***
## Re-load DataFrames into SQL

In [884]:
cleaned_adoptions.to_sql(
   "cleaned_adoptions",
   conn,
   if_exists="replace",
   index=False
)

cleaned_animals.to_sql(
   "cleaned_animals",
   conn,
   if_exists="replace",
   index=False
)

cleaned_enquiries.to_sql(
   "cleaned_enquiries",
   conn,
   if_exists="replace",
   index=False
)

cleaned_visits.to_sql(
   "cleaned_visits",
   conn,
   if_exists="replace",
   index=False
)


362

***
## Create an adoption analysis dataset

In [887]:
query="""
WITH visit_summary AS (
    SELECT
        v.enquiry_id,
        MIN(v.visit_date) AS first_visit_date,
        MAX(v.visit_date) AS last_visit_date,
        COUNT(*) AS total_visits,
        SUM(CASE WHEN UPPER(v.showed_up) = 'YES' THEN 1 ELSE 0 END) AS attended_visits
    FROM cleaned_visits v
    JOIN cleaned_enquiries e
        ON v.enquiry_id = e.enquiry_id
    GROUP BY v.enquiry_id
),
final_enquiry_dataset AS (
    SELECT
        e.enquiry_id,
        e.animal_id,
        e.customer_id,
        e.enquiry_date,
        e.channel,
        e.response_time_hours,
        e.enquiry_status,

        an.name,
        an.species,
        an.breed,
        an.age_years,
        an.colour,
        an.sex,
        an.intake_date,
        an.health_status,
        an.adoptable,

        v.first_visit_date,
        v.last_visit_date,
        v.total_visits,
        v.attended_visits,

        a.adoption_date,
        a.fee_paid,
        a.returned_within_30_days,
        a.return_date,
        CASE
            WHEN a.adoption_date IS NOT NULL THEN 1 ELSE 0
        END AS adopted

        FROM cleaned_enquiries e

        LEFT JOIN cleaned_animals an
        ON e.animal_id = an.animal_id

        LEFT JOIN visit_summary v
        ON e.enquiry_id = v.enquiry_id

        LEFT JOIN cleaned_adoptions a
        ON e.enquiry_id = a.enquiry_id
),
final_with_dates AS (
    SELECT *,
    julianday(adoption_date) - julianday(enquiry_date) AS days_enquiry_to_adoption,
    julianday(adoption_date) - julianday(first_visit_date) AS days_visit_to_adoption,
    julianday(return_date) - julianday(adoption_date) AS days_adoption_to_return
    FROM final_enquiry_dataset
)
SELECT *
FROM final_with_dates;
"""

adoption_df=pd.read_sql(query, conn)
adoption_df



,enquiry_id,animal_id,customer_id,enquiry_date,channel,response_time_hours,enquiry_status,name,species,breed,...,total_visits,attended_visits,adoption_date,fee_paid,returned_within_30_days,return_date,adopted,days_enquiry_to_adoption,days_visit_to_adoption,days_adoption_to_return
0,E0001,A115,C0369,2025-04-12 00:00:00,Walk-In,8.0,Closed,Luna,Cat,Tabby,...,1.0,0.0,None,NaN,None,None,0,NaN,NaN,NaN
1,E0002,A121,C0296,2025-03-29 00:00:00,Facebook,10.0,Open,Sasha,Cat,Domestic Longhair,...,3.0,3.0,2025-04-14 00:00:00,200.0,No,None,1,16.0,9.0,NaN
2,E0003,A095,C0288,2025-03-17 00:00:00,Website,26.0,Closed,Ruby,Cat,Maine Coon,...,NaN,NaN,None,NaN,None,None,0,NaN,NaN,NaN
3,E0004,A030,C0122,2025-02-13 00:00:00,Email,34.0,Closed,Rocky,Dog,German Shepherd,...,1.0,0.0,None,NaN,None,None,0,NaN,NaN,NaN
4,E0005,A011,C0149,2025-03-22 00:00:00,Website,7.0,Closed,Milo,Dog,Beagle,...,1.0,1.0,2025-04-07 00:00:00,100.0,No,None,1,16.0,7.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,E0316,A020,C0166,2025-04-09 00:00:00,Website,6.0,Closed,Maisie,Cat,Mixed Breed,...,1.0,0.0,None,NaN,None,None,0,NaN,NaN,NaN
316,E0317,A011,C0361,2025-03-12 00:00:00,Facebook,111.0,Closed,Milo,Dog,Beagle,...,1.0,1.0,None,NaN,None,None,0,NaN,NaN,NaN
317,E0318,A132,C0190,2025-03-16 00:00:00,Email,38.0,Closed,Poppy,Cat,Tabby,...,1.0,0.0,None,NaN,None,None,0,NaN,NaN,NaN
318,E0319,A008,C0024,2025-04-06 00:00:00,Website,NaN,Open,Pepper,Cat,Ragdoll,...,NaN,NaN,None,NaN,None,None,0,NaN,NaN,NaN


***
## Write DataFrame to csv

In [888]:
adoption_df.to_csv(
    "adoption_df.csv",
    index=False
)

***
## Initial queries to sense check data

In [889]:
adoption_df.groupby('adoptable')['health_status'].value_counts(dropna=False)

adoptable  health_status
No         Minor Issues      34
           Good              27
           In Treatment      15
           Chronic            5
           Unknown            5
Unknown    Good              17
           Minor Issues       7
           In Treatment       3
Yes        Good             135
           Minor Issues      29
           In Treatment      22
           Chronic           20
           Unknown            1
Name: count, dtype: int64

***
**Number of enquiries**

In [890]:
adoption_df['enquiry_id'].nunique()

320

**Number of visits**

In [891]:
adoption_df['first_visit_date'].notna().sum()

270

**Number of animals eventually adopted**

In [892]:
adoption_df['adopted'].sum()

105